# 7.2 Aliasing

What happens when we sample too slowly? Return to the three sinusoids that shared identical samples. From the samples alone, a 1 Hz signal and a 4 Hz signal are indistinguishable. When we undersample, high frequencies masquerade as lower ones. This phenomenon is called {vocab}`aliasing`, and the impostor frequencies are called _aliases_.

Every frequency has infinitely many aliases, spaced $f_s$ apart. For a frequency $f$, its aliases are the set

$$\text{Alias}_f = \{\, f + k \cdot f_s \mid k \in \mathbb{Z} \,\},$$

all the frequencies that differ from $f$ by an integer multiple of the sampling rate. This is exactly the copied-spectrum picture: each true frequency shows up again at every multiple of $f_s$. These aliases are not merely hard to tell apart, they are mathematically identical: a tone at $f$ and a tone at any of its aliases produce the _exact same samples_. So when the samples are reconstructed, the signal necessarily comes back at the single alias that falls within the Nyquist band $[0, f_s/2]$. We can compute that apparent frequency directly:

:::{prf:definition} Aliased frequency
:label: def-aliased-frequency
When a frequency $f$ is sampled at rate $f_s$, its apparent (aliased) frequency is

$$f_{\text{alias}} = \min\big(f \bmod f_s,\; f_s - (f \bmod f_s)\big),$$

which always lies in $[0, f_s/2]$. If $f$ is already in $[0, f_s/2]$, then $f_{\text{alias}} = f$ and no aliasing occurs. Otherwise $f_{\text{alias}} \neq f$.
:::

## Aliasing in practice

**Aliasing is a very real, audible phenomenon, not just a theoretical construct.** To hear it, we can synthesize a tone whose frequency slowly sweeps up from 220 Hz to 880 Hz and back, at a few different sample rates. The following clips were each synthesized directly at the given $f_s$ (then resampled purely for playback), so any aliasing is baked into the sound:

:::{audio-list}
{audio}`Sweep at f_s = 2000 Hz <./assets/audio-alias-2000.wav>`

{audio}`Sweep at f_s = 1000 Hz <./assets/audio-alias-1000.wav>`

{audio}`Sweep at f_s = 500 Hz <./assets/audio-alias-500.wav>`

The same 220-to-880 Hz pitch sweep synthesized at three sample rates. At $f_s = 2000$ Hz the sweep is clean. As $f_s$ drops, the upper part of the sweep exceeds the Nyquist frequency and folds back down, so the pitch audibly reverses direction.
:::

The figure below plots what is happening. The true frequency (blue) rises above the Nyquist frequency (red) once $f_s$ is small enough, and the frequency we actually hear (orange) folds back below it:

:::{figure}
![Three side-by-side plots of the pitch sweep at sample rates 2000, 1000, and 500 Hz. Each shows the true frequency rising and falling as a smooth hump, a horizontal Nyquist line at f_s over 2, and the heard (aliased) frequency. At 2000 Hz the heard frequency tracks the true one. At 1000 and 500 Hz the true frequency crosses the Nyquist line and the heard frequency folds back downward, once at 1000 Hz and twice at 500 Hz.](./assets/fig-aliasing-practice.png)

The pitch sweep at three sample rates. When the true frequency (blue) crosses the Nyquist frequency $f_s/2$ (red), the heard frequency (orange) reflects back downward. The full sonification code is available as an interactive example below.
:::

For frequencies just above the Nyquist frequency, in the range $[f_s/2, f_s]$, this reflection is colloquially called {vocab}`foldover`, because the aliased frequencies mirror back across the Nyquist frequency as if it were a crease in a folded sheet of paper.

You can explore this yourself. The interactive example below lets you set the sample rate and the pitch contour, then synthesizes and plays the result so you can hear aliasing emerge as you lower $f_s$:

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq

PLAYBACK_SR = 44100


def aliased(f, f_s):
    m = np.mod(f, f_s)
    return np.minimum(m, f_s - m)


def show_aliasing(f_s, control_points):
    # control_points: list of (time_seconds, frequency_hz), interpolated
    # linearly in frequency.
    times = [p[0] for p in control_points]
    freqs = [p[1] for p in control_points]
    dur = times[-1]

    n = np.arange(int(dur * f_s))
    freq = np.interp(n / f_s, times, freqs)

    # synthesize at f_s by accumulating phase (Chapter 6), resample for playback
    x = np.sin(np.cumsum(2 * np.pi * freq / f_s))
    audio = pq.Audio(x.astype(np.float32), int(f_s)).resample(PLAYBACK_SR)

    t = n / f_s
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(t, freq, color="#007BC0", label="true frequency")
    ax.plot(t, aliased(freq, f_s), color="#FDB515", ls="--", label="heard (aliased)")
    ax.axhline(f_s / 2, color="#C41230", lw=1.3, label="Nyquist $f_s/2$")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_ylim(0, max(freq.max(), f_s / 2) * 1.15)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()
    return pq.play(audio)


In [ ]:
# Edit the sample rate and the pitch contour, then run to see and hear aliasing.
# Each control point is (time in seconds, frequency in Hz).
f_s = 500  # sampling rate (Hz): lower it to force aliasing

pitch_contour = [
    (0.0, 220.0),
    (1.0, 220.0),
    (6.0, 880.0),
    (7.0, 880.0),
    (12.0, 220.0),
    (13.0, 220.0),
]

show_aliasing(f_s, pitch_contour)


## Aliasing beyond audio

Aliasing is not unique to digital audio. It arises whenever any signal is sampled too slowly, including the sampling of _light_ that our eyes and cameras perform. A classic example is the {vocab}`wagon-wheel effect`, in which the spoked wheels of a moving vehicle appear to slow down, stop, or even spin backwards on film. A camera captures frames at a fixed rate (its sampling rate). When a wheel rotates by nearly a full spoke-spacing between frames, its true rotation aliases to a much slower apparent rotation, and when it rotates by slightly more than a spoke-spacing, the alias runs backwards (a negative frequency):

In [ ]:
# hide
import numpy as np
from manim import *

import icm_anim as anim


In [ ]:
# hide
class WagonWheel(Scene):
    """A spoked wheel whose true rotation speeds up steadily, filmed at a
    fixed frame rate. As the per-frame rotation approaches and exceeds half a
    spoke-spacing, the wheel appears to slow, stop, and spin backwards: the
    wagon-wheel effect, an everyday example of aliasing."""

    def construct(self):
        R = 2.5
        n_spokes = 8
        theta = ValueTracker(0.0)  # total true rotation (radians)

        def spoke_end(angle):
            return R * np.array([np.cos(angle), np.sin(angle), 0.0])

        def wheel():
            g = VGroup(Circle(radius=R, color=anim.IRON, stroke_width=3.5))
            a = theta.get_value()
            for k in range(n_spokes):
                ang = a + k * TAU / n_spokes
                is_marker = (k == 0)
                g.add(Line(
                    ORIGIN, spoke_end(ang),
                    color=anim.RED if is_marker else anim.IRON,
                    stroke_width=6 if is_marker else 2.5,
                ))
            g.add(Dot(spoke_end(a), color=anim.RED, radius=0.13))
            return g

        title = MathTex(r"\text{a spoked wheel, filmed at a fixed frame rate}")
        title.scale(0.7).to_edge(UP, buff=0.4)
        hint = MathTex(r"\text{true speed increases} \longrightarrow")
        hint.scale(0.6).to_edge(DOWN, buff=0.5).set_color(anim.IRON)

        self.add(always_redraw(wheel), title, hint)
        # Accelerate the true rotation quadratically so the apparent motion
        # sweeps through forward, frozen, and reversed regimes.
        self.play(theta.animate.set_value(TAU * 22), run_time=10,
                  rate_func=lambda t: t * t)
        self.wait(0.4)


anim.show(WagonWheel)


You may have experienced the same effect at a concert with a strobe light. The strobe flashes at a fixed rate, sampling the motion of the dancers, and this can make movements appear frozen, slowed, or reversed. The figure below shows a dancer bobbing up and down once per second (a 1 Hz motion). The leftmost panel is the continuous motion, and the other four "sample" it by holding whichever frame was most recently caught by a strobe at the labeled rate, while a shared clock advances:

:::{figure}
![Five side-by-side copies of the same dancer with a shared time counter at the top. Left to right: the continuous motion x(t), then sampled versions at f_s = 4 Hz (oversampled), 2 Hz (critically sampled), 4/3 Hz (foldover), and 1 Hz (aliased to 0 Hz). As the clock advances, the continuous and oversampled dancers move smoothly, the 1 Hz dancer stays frozen, and the 4/3 Hz dancer drifts backwards.](./assets/fig-strobe-dance.gif)

The same 1 Hz dance, continuous (left) and sampled at four rates. At $f_s = 4$ Hz the motion still looks correct (oversampled). At $f_s = 2$ Hz it collapses to just two alternating poses (critical sampling). At $f_s = 1$ Hz the dancer is caught at the same phase every time and appears frozen, aliased all the way to 0 Hz. At $f_s = \tfrac{4}{3}$ Hz the dancer appears to drift slowly _backwards_, the same foldover we saw with sound.
:::

## Critical sampling

The sampling theorem demands $f_s > 2 f_{\max}$, a _strict_ inequality. What happens right at the boundary, when $f_s = 2 f_{\max}$? This edge case is called {vocab}`critical sampling`, and it turns out to be genuinely ambiguous.

Consider a cosine at exactly the Nyquist frequency, $x(t) = \cos(2\pi t)$ with $f_{\max} = 1$ Hz, sampled at $f_s = 2$ Hz. The samples are

$$x[n] = \cos(\pi n) = [\,1, -1, 1, -1, \ldots\,],$$

a perfectly reasonable representation of a 1 Hz tone. But now consider a _sine_ at the same frequency, $x(t) = \sin(2\pi t)$, sampled at the same rate. Its samples are

$$x[n] = \sin(\pi n) = [\,0, 0, 0, 0, \ldots\,].$$

Every sample lands exactly on a zero crossing, so the sine vanishes completely. Those all-zero samples could equally well represent silence, a signal at 0 Hz. At critical sampling, then, a component at exactly $f_s/2$ may or may not survive, depending on its phase. This is precisely why the theorem requires a strict inequality: at the boundary, reconstruction is no longer guaranteed.